In [ ]:
# ── Core ML / DL ───────────────────────────────────────────
!pip install -q tensorflow-datasets sentencepiece editdistance jiwer

# ── NLP / Transformers ─────────────────────────────────────
!pip install -q transformers torch datasets langdetect deep-translator sacremoses

# ── Vision ─────────────────────────────────────────────────
!pip install -q albumentations opencv-python-headless Pillow

# ── OCR Engines: Tesseract + language packs ─────────────────
# We install English, Hindi (for Devanagari/Marathi), Marathi, and Tamil
!apt-get install -y -qq tesseract-ocr tesseract-ocr-hin tesseract-ocr-mar tesseract-ocr-tam tesseract-ocr-tel
!pip install -q pytesseract

# ── Analysis / Visualisation ───────────────────────────────
!pip install -q scikit-learn seaborn matplotlib nltk spacy scipy
!python -m spacy download en_core_web_sm -q

print("[OK] All dependencies installed!")



In [ ]:
# ============================================================
# SECTION 2 — CONFIGURATION (single source of truth)
# ============================================================

# ══════════════════════════════════════════════════════════════
#  GLOBAL CONFIGURATION — edit only this cell to tune the system
# ══════════════════════════════════════════════════════════════

import os, re, time, warnings, json, pickle, random, functools
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter, defaultdict
warnings.filterwarnings('ignore')

# ── Debug flag ─────────────────────────────────────────────
DEBUG = True          # Set False to suppress intermediate visualizations

# ── Paths ──────────────────────────────────────────────────
CKPT_DIR    = Path('/content/htr_checkpoints')
CKPT_DIR.mkdir(exist_ok=True)
CNN_CKPT    = CKPT_DIR / 'cnn_model'
SP_MODEL    = CKPT_DIR / 'spm.model'
HIST_FILE   = CKPT_DIR / 'train_history.pkl'
DATA_CACHE  = CKPT_DIR / 'multilingual_corpus.pkl'
SYNTH_CACHE = CKPT_DIR / 'synth_word_images.pkl'

# ── Model / Training ───────────────────────────────────────
SEED        = 42
IMG_H       = 28
IMG_W       = 28
NUM_CLASSES = 26
BATCH_SIZE  = 128
CNN_EPOCHS  = 25
WARMUP_STEPS = 500
CHARSET     = list('abcdefghijklmnopqrstuvwxyz')
BLANK_IDX   = NUM_CLASSES         # CTC blank token index
SP_VOCAB_SIZE = 734               # SentencePiece vocabulary size

# ── Preprocessing ──────────────────────────────────────────
CLAHE_CLIP_LIMIT    = 2.0         # Higher = more contrast; try 3.0 for gray backgrounds
CLAHE_TILE_SIZE     = (8, 8)
BILATERAL_D         = 5           # Bilateral filter neighbourhood size
BILATERAL_SIGMA     = 50          # Bilateral filter sigma (colour + space)
THRESH_BLOCK_SIZE   = 21          # Adaptive threshold block size (must be odd)
THRESH_C            = 5           # Adaptive threshold constant subtracted from mean
MORPH_KERNEL_SIZE   = (2, 2)      # Morphological open kernel (noise removal)

# ── Line Segmentation ──────────────────────────────────────
LINE_DILATION_H_RATIO = 0.02      # Kernel height as fraction of image height
LINE_DILATION_W_RATIO = 0.03      # Kernel width as fraction of image width
MIN_LINE_AREA_RATIO   = 0.0005    # Minimum contour area (fraction of image) to be a line
MIN_LINE_HEIGHT_RATIO = 0.005
MIN_LINE_WIDTH_RATIO  = 0.10
LINE_MERGE_OVERLAP    = 0.05      # Fraction of overlap to merge adjacent lines

# ── Word Segmentation — Connected Components ────────────────
CC_MIN_COMPONENT_AREA     = 20    # Ignore components smaller than this (noise)
CC_MAX_COMPONENT_AREA_RATIO = 0.8 # Ignore components larger than this (% of line)
CC_MAX_HORIZONTAL_GAP_PX  = 20    # Max pixel gap to group adjacent CCs into a word
CC_MIN_WORD_WIDTH_PX      = 5     # Minimum word box width (pixels)
CC_MAX_WORD_HEIGHT_RATIO  = 0.9   # Ignore components taller than 90% of line height

# ── Word Segmentation — X-Projection ───────────────────────
XPROJ_MIN_WORD_WIDTH_RATIO   = 0.01  # Minimum word width as fraction of line width
XPROJ_WORD_GAP_SENSITIVITY   = 0.20  # Lower = more sensitive to gaps (more splits)
XPROJ_SMOOTHING_KERNEL_SIZE  = 7     # Gaussian kernel size for profile smoothing

# ── Word-level Merging ─────────────────────────────────────
BBOX_IOU_MERGE_THRESHOLD = 0.30  # IOU threshold to merge overlapping word boxes
WORD_PADDING_PX          = 4     # Pixels of padding added around each word crop

# ── Language Detection ─────────────────────────────────────
# Unicode block ranges used for fast heuristic detection
DEVANAGARI_RANGE = (0x0900, 0x097F)  # Hindi, Marathi
TAMIL_RANGE      = (0x0B80, 0x0BFF)
# Minimum fraction of chars in script to classify as that script
SCRIPT_DETECT_THRESHOLD = 0.30

# ── OCR ────────────────────────────────────────────────────
TESSERACT_PSM   = 8    # Single word mode (7 = single line; 6 = block)
TESSERACT_OEM   = 1    # LSTM engine
CNN_FALLBACK_ASPECT_RATIO = 1.5  # If word_w/word_h < this, allow CNN fallback

# ── Corpus ─────────────────────────────────────────────────
TARGET_PER_LANG = 600

# ── Evaluation ─────────────────────────────────────────────
TTA_N_PASSES = 8
EVAL_SAMPLE_N = 3000

# ── Reproducibility ────────────────────────────────────────
np.random.seed(SEED)
random.seed(SEED)

print("[OK] Configuration loaded.")
print(f"  DEBUG mode     : {DEBUG}")
print(f"  Checkpoint dir : {CKPT_DIR}")
print(f"  Charset size   : {len(CHARSET)}")

In [ ]:

# ============================================================
# SECTION 3 — IMPORTS
# ============================================================
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, mixed_precision
import tensorflow_datasets as tfds
from sklearn.metrics import confusion_matrix, classification_report, f1_score
import seaborn as sns
import cv2
from PIL import Image as PILImage, ImageDraw, ImageFont
import editdistance
import scipy.signal
import pytesseract

import nltk
for pkg in ['punkt', 'stopwords', 'wordnet', 'averaged_perceptron_tagger', 'punkt_tab']:
    nltk.download(pkg, quiet=True)
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import spacy
nlp_spacy = spacy.load('en_core_web_sm')

from langdetect import detect, detect_langs, DetectorFactory
DetectorFactory.seed = SEED
from deep_translator import GoogleTranslator

# ── Mixed precision ────────────────────────────────────────
policy = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(policy)

tf.random.set_seed(SEED)

print(f"[OK] TF version  : {tf.__version__}")
print(f"[OK] GPU devices : {tf.config.list_physical_devices('GPU')}")
print(f"[OK] Mixed prec  : {mixed_precision.global_policy().name}")
print("[OK] All imports done!")

In [ ]:

# ============================================================
# SECTION 4 — VISUALIZATION MODULE (centralized)
# ============================================================
# ══════════════════════════════════════════════════════════════
#  CENTRALIZED VISUALIZATION MODULE
#  Every stage imports from this namespace — do not scatter plt calls.
# ══════════════════════════════════════════════════════════════

def show_image(title, image, cmap='gray', figsize=(8, 3)):

    if not DEBUG:
        return
    plt.figure(figsize=figsize)
    if len(image.shape) == 3 and image.shape[2] == 3:
        # Assume BGR (OpenCV) — convert to RGB for matplotlib
        display = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        plt.imshow(display)
    else:
        plt.imshow(image, cmap=cmap)
    plt.title(title, fontsize=12, fontweight='bold')
    plt.axis('off')
    plt.tight_layout()
    plt.show()


def show_before_after(title_before, img_before, title_after, img_after,
                      cmap='gray', figsize=(14, 3)):

    if not DEBUG:
        return
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    for ax, title, img in zip(axes, [title_before, title_after], [img_before, img_after]):
        disp = img
        if len(img.shape) == 3 and img.shape[2] == 3:
            disp = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            ax.imshow(disp)
        else:
            ax.imshow(disp, cmap=cmap)
        ax.set_title(title, fontsize=11, fontweight='bold')
        ax.axis('off')
    plt.tight_layout()
    plt.show()


def show_bounding_boxes(image, boxes, color=(0, 200, 0), thickness=2,
                        labels=None, title='Bounding Boxes', figsize=(12, 4)):

    if not DEBUG:
        return
    if len(image.shape) == 2:
        canvas = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
    else:
        canvas = image.copy()

    for i, (x1, y1, x2, y2) in enumerate(boxes):
        cv2.rectangle(canvas, (x1, y1), (x2, y2), color, thickness)
        if labels and i < len(labels):
            # Small label above the box
            cv2.putText(canvas, str(labels[i]), (x1, max(0, y1 - 4)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1, cv2.LINE_AA)

    plt.figure(figsize=figsize)
    plt.imshow(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB))
    plt.title(f'{title}  ({len(boxes)} boxes)', fontsize=12, fontweight='bold')
    plt.axis('off')
    plt.tight_layout()
    plt.show()


def show_projection_profile(binary_image, axis='x', title='Projection Profile'):

    if not DEBUG:
        return
    if axis == 'x':
        profile = np.sum(binary_image > 0, axis=0)
        xlabel  = 'Column (X)'
    else:
        profile = np.sum(binary_image > 0, axis=1)
        xlabel  = 'Row (Y)'

    fig, axes = plt.subplots(1, 2, figsize=(14, 3),
                             gridspec_kw={'width_ratios': [2, 1]})
    axes[0].plot(profile, color='steelblue', linewidth=1)
    axes[0].fill_between(range(len(profile)), profile, alpha=0.3, color='steelblue')
    axes[0].set_title(f'{title} ({axis.upper()}-projection)', fontweight='bold')
    axes[0].set_xlabel(xlabel)
    axes[0].set_ylabel('Black pixel count')
    axes[0].grid(alpha=0.3)

    axes[1].imshow(binary_image, cmap='gray')
    axes[1].set_title('Binary Image', fontweight='bold')
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()


def show_word_crops(crops_gray, predictions=None, max_show=12, title='Word Crops'):

    if not DEBUG:
        return
    n = min(len(crops_gray), max_show)
    if n == 0:
        print(f'  [VIZ] No crops to display for "{title}"')
        return
    fig, axes = plt.subplots(1, n, figsize=(n * 2.5, 2.5))
    if n == 1:
        axes = [axes]
    for i, (ax, crop) in enumerate(zip(axes, crops_gray[:n])):
        ax.imshow(crop, cmap='gray')
        if predictions and i < len(predictions):
            p = predictions[i]
            label = f"{p.get('text', '?')}\\n{p.get('lang', '')} {p.get('conf', 0):.2f}"
            ax.set_title(label, fontsize=7)
        ax.axis('off')
    plt.suptitle(f'{title}  (showing {n} of {len(crops_gray)})',
                 fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.show()


def show_multi_stage(stages, figsize=(16, 3)):

    if not DEBUG:
        return
    n = len(stages)
    fig, axes = plt.subplots(1, n, figsize=figsize)
    if n == 1:
        axes = [axes]
    for ax, (title, img) in zip(axes, stages):
        if len(img.shape) == 3 and img.shape[2] == 3:
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        else:
            ax.imshow(img, cmap='gray')
        ax.set_title(title, fontsize=9, fontweight='bold')
        ax.axis('off')
    plt.tight_layout()
    plt.show()


def show_language_boxes(image, word_results, figsize=(12, 4)):
 
    if not DEBUG:
        return
    LANG_COLORS = {
        'english':    (0,   200,  0),    # green
        'marathi':    (255,  80,  0),    # orange
        'tamil':      (0,   100, 255),   # blue
        'unknown':    (180, 180, 180),   # gray
    }
    if len(image.shape) == 2:
        canvas = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
    else:
        canvas = image.copy()

    legend_patches = []
    seen_langs = set()
    for r in word_results:
        x1, y1, x2, y2 = r['bbox']
        lang  = r.get('lang', 'unknown').lower()
        color = LANG_COLORS.get(lang, (180, 180, 180))
        cv2.rectangle(canvas, (x1, y1), (x2, y2), color, 2)
        cv2.putText(canvas, r.get('text', '?')[:8], (x1, max(0, y1 - 4)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)
        if lang not in seen_langs:
            seen_langs.add(lang)
            legend_patches.append(
                mpatches.Patch(color=[c/255 for c in color], label=lang.capitalize()))

    plt.figure(figsize=figsize)
    plt.imshow(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB))
    plt.title('Language Detection — Colour-Coded Bounding Boxes',
              fontsize=12, fontweight='bold')
    if legend_patches:
        plt.legend(handles=legend_patches, loc='upper right', fontsize=9)
    plt.axis('off')
    plt.tight_layout()
    plt.show()


print("[OK] Visualization module loaded.")
print("  Functions: show_image | show_before_after | show_bounding_boxes")
print("             show_projection_profile | show_word_crops | show_multi_stage")
print("             show_language_boxes")


In [ ]:

# ============================================================
# SECTION 5 — PREPROCESSING MODULE
# ============================================================
# ══════════════════════════════════════════════════════════════
#  PREPROCESSING MODULE
#  All functions return named intermediate variables so callers
#  can inspect and visualize any stage.
# ══════════════════════════════════════════════════════════════

def load_image(image_path):

    img_bgr = cv2.imread(str(image_path))
    if img_bgr is None:
        raise FileNotFoundError(
            f"[PREPROCESS] Cannot read image at: {image_path}\\n"
            f"  Check that the file exists and is a valid image format."
        )
    img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    print(f"  [PREPROCESS] Loaded: {image_path}  shape={img_bgr.shape}")
    return img_bgr, img_gray


def upscale_if_small(img_gray, min_height=80, scale_factor=2):

    h, w = img_gray.shape
    if h < min_height:
        # WHY: Tesseract needs at least ~50px character height to work reliably
        new_h = h * scale_factor
        new_w = w * scale_factor
        scaled = cv2.resize(img_gray, (new_w, new_h), interpolation=cv2.INTER_CUBIC)
        print(f"  [PREPROCESS] Upscaled: {h}×{w} → {new_h}×{new_w} (factor={scale_factor})")
        return scaled, scale_factor
    return img_gray, 1


def enhance_contrast(img_gray):

    # CLAHE: adaptive histogram equalization in local tiles
    clahe    = cv2.createCLAHE(
        clipLimit=CLAHE_CLIP_LIMIT,
        tileGridSize=CLAHE_TILE_SIZE
    )
    enhanced = clahe.apply(img_gray)

    # Bilateral filter: edge-preserving smoothing
    denoised = cv2.bilateralFilter(
        enhanced,
        d=BILATERAL_D,
        sigmaColor=BILATERAL_SIGMA,
        sigmaSpace=BILATERAL_SIGMA
    )
    return enhanced, denoised


def binarize(denoised):

    binary = cv2.adaptiveThreshold(
        denoised,
        maxValue=255,
        adaptiveMethod=cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        thresholdType=cv2.THRESH_BINARY_INV,   # inverted: text=white, bg=black
        blockSize=THRESH_BLOCK_SIZE,
        C=THRESH_C
    )
    # Morphological open: remove tiny noise specks
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, MORPH_KERNEL_SIZE)
    binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel, iterations=1)
    return binary


def preprocess_image(image_path, debug=None):

    if debug is None:
        debug = DEBUG

    print("\\n[PREPROCESS] Starting preprocessing pipeline...")

    # Step 1: Load
    img_bgr, img_gray = load_image(image_path)

    # Step 2: Upscale if tiny
    img_upscaled, scale = upscale_if_small(img_gray)

    # Step 3: Contrast enhancement
    img_enhanced, img_denoised = enhance_contrast(img_upscaled)

    # Step 4: Binarize
    img_binary = binarize(img_denoised)

    # ── Store all intermediates for debugging ────────────────
    intermediates = {
        'original_bgr':  img_bgr,
        'original_gray': img_gray,
        'upscaled':      img_upscaled,
        'enhanced':      img_enhanced,
        'denoised':      img_denoised,
        'binary':        img_binary,
        'scale_factor':  scale,
    }

    # ── Debug visualizations ─────────────────────────────────
    if debug:
        print("  [DEBUG] Showing all preprocessing stages:")
        show_multi_stage([
            ('1. Original (gray)',   img_upscaled),
            ('2. CLAHE Enhanced',   img_enhanced),
            ('3. Bilateral Denoise', img_denoised),
            ('4. Binary (inverted)', img_binary),
        ], figsize=(18, 3))

        # Also show projection profile so we can anticipate line/word gaps
        show_projection_profile(img_binary, axis='y', title='Row Profile (line gaps)')

    print(f"  [PREPROCESS] Done. Binary shape: {img_binary.shape}")
    return intermediates


# ── Devanagari shirorekha removal (optional, helps Marathi/Hindi segmentation) ──
def remove_shirorekha(binary_img):

    # Wide horizontal kernel targets the continuous horizontal stroke
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 1))
    return cv2.erode(binary_img, kernel, iterations=1)


print("[OK] Preprocessing module loaded.")
print("  Functions: load_image | upscale_if_small | enhance_contrast")
print("             binarize | preprocess_image | remove_shirorekha")


In [ ]:
# ============================================================
# SECTION 6 — SEGMENTATION MODULE
# ============================================================
# ══════════════════════════════════════════════════════════════
#  SEGMENTATION MODULE — Line Segmentation
# ══════════════════════════════════════════════════════════════

def find_text_lines(binary_img, debug=None):

    if debug is None:
        debug = DEBUG

    h, w = binary_img.shape

    # ── Step 1: Dilate to merge characters within a line ────────────────────
    # WHY: Characters in the same line are vertically aligned but may not
    # physically touch. Dilation bridges these gaps before contour detection.
    kernel_h = max(3, int(h * LINE_DILATION_H_RATIO))
    kernel_w = max(3, int(w * LINE_DILATION_W_RATIO))
    kernel   = cv2.getStructuringElement(cv2.MORPH_RECT, (kernel_w, kernel_h))
    dilated  = cv2.dilate(binary_img, kernel, iterations=1)

    # ── Step 2: Find contours on dilated image ───────────────────────────────
    contours, _ = cv2.findContours(dilated, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # ── Step 3: Filter by minimum area / dimensions ──────────────────────────
    line_boxes = []
    for contour in contours:
        x, y, cw, ch = cv2.boundingRect(contour)
        area = cw * ch
        # Reject if too small, too short, or too narrow
        if (area  > w * h * MIN_LINE_AREA_RATIO and
            ch    > h * MIN_LINE_HEIGHT_RATIO   and
            cw    > w * MIN_LINE_WIDTH_RATIO):
            line_boxes.append((x, y, x + cw, y + ch))

    # Sort top-to-bottom (by y1 coordinate)
    line_boxes.sort(key=lambda b: b[1])

    # ── Step 4: Merge overlapping or nearly-touching lines ───────────────────
    merged = []
    if line_boxes:
        cur = list(line_boxes[0])
        for nxt in line_boxes[1:]:
            # Check if next line overlaps or is very close to current
            overlap_margin = max(5, int(
                min(cur[3] - cur[1], nxt[3] - nxt[1]) * LINE_MERGE_OVERLAP))
            if nxt[1] <= cur[3] + overlap_margin:
                # Extend current box to include next
                cur[0] = min(cur[0], nxt[0])
                cur[1] = min(cur[1], nxt[1])
                cur[2] = max(cur[2], nxt[2])
                cur[3] = max(cur[3], nxt[3])
            else:
                merged.append(tuple(cur))
                cur = list(nxt)
        merged.append(tuple(cur))

    intermediates = {'dilated': dilated}

    # ── Debug visualizations ─────────────────────────────────────────────────
    if debug:
        show_before_after(
            'Binary (input to line seg)',    binary_img,
            'Dilated (for line detection)',  dilated,
            title_before='Binary', title_after='Dilated'
        )
        show_bounding_boxes(
            binary_img, merged,
            color=(0, 200, 255),   # yellow-green
            title=f'Line Segmentation ({len(merged)} lines)'
        )
        # Y-projection shows row-level ink density → gaps between lines are visible
        show_projection_profile(binary_img, axis='y', title='Y-Projection — Line Gaps')

    print(f"  [LINE SEG] Detected {len(merged)} text lines")
    return merged, intermediates


In [ ]:

# ══════════════════════════════════════════════════════════════
#  SEGMENTATION MODULE — Word Segmentation (Dual Method)
# ══════════════════════════════════════════════════════════════

# ── Method A: Connected Components ──────────────────────────

def word_seg_connected_components(binary_line, debug=None):

    if debug is None:
        debug = DEBUG

    h, w = binary_line.shape
    if w == 0 or h == 0:
        return []

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
        binary_line, connectivity=8)

    # Collect valid components
    components = []
    for i in range(1, num_labels):   # skip background (label 0)
        x, y, cw, ch, area = stats[i]
        # WHY these filters: small area = noise; very large = image artifact
        if (area < CC_MIN_COMPONENT_AREA or
                area > h * w * CC_MAX_COMPONENT_AREA_RATIO):
            continue
        # Filter components taller than 90% of line height (usually not characters)
        if ch > h * CC_MAX_WORD_HEIGHT_RATIO:
            continue
        components.append((x, y, x + cw, y + ch))

    if not components:
        return []

    # Sort left to right
    components.sort(key=lambda c: c[0])

    # Group components into words based on horizontal proximity
    word_boxes = []
    cur = list(components[0])
    for (x1, y1, x2, y2) in components[1:]:
        # If this component starts within the gap threshold of the current group
        if x1 - cur[2] < CC_MAX_HORIZONTAL_GAP_PX:
            # Extend current group's bounding box
            cur[2] = max(cur[2], x2)
            cur[3] = max(cur[3], y2)
            cur[1] = min(cur[1], y1)
        else:
            # Gap is too large — start a new word
            if cur[2] - cur[0] > CC_MIN_WORD_WIDTH_PX:
                word_boxes.append(tuple(cur))
            cur = [x1, y1, x2, y2]
    if cur[2] - cur[0] > CC_MIN_WORD_WIDTH_PX:
        word_boxes.append(tuple(cur))

    # Final filter: discard boxes too wide (likely the whole line) or too thin
    word_boxes = [
        b for b in word_boxes
        if (b[2] - b[0]) > CC_MIN_WORD_WIDTH_PX
        and (b[2] - b[0]) < w * 0.9
        and (b[3] - b[1]) > h * 0.1
    ]

    if debug:
        show_bounding_boxes(binary_line, word_boxes,
                            color=(0, 200, 0),
                            title=f'CC Word Boxes ({len(word_boxes)})')
    return word_boxes


# ── Method B: X-Projection ───────────────────────────────────

def word_seg_xprojection(binary_line, debug=None):

    if debug is None:
        debug = DEBUG

    h, w = binary_line.shape
    if w == 0 or h == 0:
        return []

    # ── Step 1: Column projection ────────────────────────────────────────────
    # Count black pixels per column (binary_line: text=255, bg=0)
    projection = np.sum(binary_line > 0, axis=0).astype(float)

    # ── Step 2: Smooth to reduce noise in projection ─────────────────────────
    k = XPROJ_SMOOTHING_KERNEL_SIZE
    if k > 1:
        projection = scipy.signal.convolve(
            projection, np.ones(k) / k, mode='same')

    # ── Step 3: Normalize ────────────────────────────────────────────────────
    proj_min, proj_max = projection.min(), projection.max()
    if proj_max - proj_min == 0:
        return []   # Flat profile = blank line
    norm_proj = (projection - proj_min) / (proj_max - proj_min)

    # ── Step 4: Find valleys (gap columns) ───────────────────────────────────
    # WHY adaptive: a fixed threshold fails on images with different ink densities.
    # Using a fraction of the mean makes it scale with the actual text density.
    threshold = norm_proj.mean() * XPROJ_WORD_GAP_SENSITIVITY
    is_gap = norm_proj < threshold

    # ── Step 5: Extract word regions between gaps ────────────────────────────
    boundaries = []
    in_gap = False
    for i, gap in enumerate(is_gap):
        if gap and not in_gap:
            boundaries.append(i)
            in_gap = True
        elif not gap and in_gap:
            boundaries.append(i)
            in_gap = False
    if in_gap:
        boundaries.append(w)

    word_boxes = []
    start = 0
    for i, boundary in enumerate(boundaries):
        if i % 2 == 0:   # Start of a gap
            if boundary > start:
                bw = boundary - start
                if bw > w * XPROJ_MIN_WORD_WIDTH_RATIO:
                    word_boxes.append((start, 0, boundary, h))
            start = boundary
        else:             # End of a gap
            start = boundary
    if start < w:
        bw = w - start
        if bw > w * XPROJ_MIN_WORD_WIDTH_RATIO:
            word_boxes.append((start, 0, w, h))

    word_boxes = [(x1, y1, x2, y2) for (x1, y1, x2, y2) in word_boxes if x2 - x1 > 0]

    if debug:
        # Show projection + boxes together
        fig, axes = plt.subplots(2, 1, figsize=(15, 5),
                                 gridspec_kw={'height_ratios': [1, 3]})
        axes[0].plot(norm_proj, color='steelblue', linewidth=1)
        axes[0].axhline(threshold, color='red', linestyle='--',
                        label=f'Valley threshold ({threshold:.2f})')
        axes[0].set_title('X-Projection Profile (normalized)', fontweight='bold')
        axes[0].legend()
        axes[0].set_xlim(0, w)
        axes[0].grid(alpha=0.3)

        axes[1].imshow(binary_line, cmap='gray')
        for (x1, y1, x2, y2) in word_boxes:
            rect = mpatches.Rectangle(
                (x1, y1), x2 - x1, y2 - y1,
                fill=False, edgecolor='cyan', linewidth=2)
            axes[1].add_patch(rect)
        axes[1].set_title(f'X-Projection Word Boxes ({len(word_boxes)})', fontweight='bold')
        axes[1].axis('off')
        plt.tight_layout(); plt.show()

    return word_boxes


# ── Merge two sets of bounding boxes ─────────────────────────

def merge_word_boxes(boxes_a, boxes_b, iou_threshold=None):

    if iou_threshold is None:
        iou_threshold = BBOX_IOU_MERGE_THRESHOLD

    all_boxes = list(boxes_a) + list(boxes_b)
    if not all_boxes:
        return []
    all_boxes.sort(key=lambda b: b[0])

    def iou(a, b):
        ix1 = max(a[0], b[0]); iy1 = max(a[1], b[1])
        ix2 = min(a[2], b[2]); iy2 = min(a[3], b[3])
        inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
        union = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
        return inter / max(1, union)

    merged = []
    for box in all_boxes:
        added = False
        for i, mbox in enumerate(merged):
            if iou(box, mbox) > iou_threshold:
                # Union: take the outer boundary
                merged[i] = (
                    min(box[0], mbox[0]), min(box[1], mbox[1]),
                    max(box[2], mbox[2]), max(box[3], mbox[3])
                )
                added = True
                break
        if not added:
            merged.append(box)

    merged.sort(key=lambda b: b[0])
    return merged


print("[OK] Segmentation module loaded.")
print("  Functions: find_text_lines | word_seg_connected_components")
print("             word_seg_xprojection | merge_word_boxes")


In [ ]:

# ============================================================
# SECTION 7 — LANGUAGE DETECTION MODULE
# ============================================================
# ══════════════════════════════════════════════════════════════
#  LANGUAGE DETECTION MODULE
# ══════════════════════════════════════════════════════════════

# Unicode block ranges for each script we support
SCRIPT_RANGES = {
    'marathi':  (DEVANAGARI_RANGE,),   # Devanagari covers Hindi + Marathi
    'tamil':    (TAMIL_RANGE,),
    # 'telugu': ((0x0C00, 0x0C7F),),  # Uncomment to add Telugu
    # 'kannada': ((0x0C80, 0x0CFF),), # Uncomment to add Kannada
}


def detect_script_heuristic(text):

    if not text or not text.strip():
        return 'unknown'

    total_chars = len([c for c in text if c.strip()])
    if total_chars == 0:
        return 'unknown'

    for lang, ranges in SCRIPT_RANGES.items():
        in_range = sum(
            1 for c in text
            if any(lo <= ord(c) <= hi for (lo, hi) in ranges)
        )
        fraction = in_range / total_chars
        if fraction >= SCRIPT_DETECT_THRESHOLD:
            return lang

    # If no non-Latin script detected, use langdetect for Latin sub-languages
    return 'english'   # default to English for Latin script


def detect_language_langdetect(text):

    if not text or len(text.strip()) < 3:
        return 'unknown', 0.0
    try:
        probs = detect_langs(text)
        top   = probs[0]
        lang_map = {
            'en': 'english', 'hi': 'marathi',   # hindi OCR often returns 'hi'
            'mr': 'marathi', 'ta': 'tamil',
        }
        lang_name = lang_map.get(top.lang, 'english')
        return lang_name, float(top.prob)
    except Exception:
        return 'unknown', 0.0


def detect_word_language(word_text, word_crop_gray=None):

    # ── Path 1: Unicode heuristic on the OCR text ──────────────────────────
    if word_text and word_text.strip():
        lang = detect_script_heuristic(word_text)
        if lang != 'english':
            # Found a non-Latin script — high confidence
            return lang, 0.90, 'unicode-heuristic'

    # ── Path 2: Visual heuristic on the image crop ─────────────────────────
    if word_crop_gray is not None:
        if _is_devanagari_visual(word_crop_gray):
            return 'marathi', 0.70, 'visual-shirorekha'

    # ── Path 3: Statistical detection (langdetect) for Latin script ─────────
    if word_text and word_text.strip():
        lang, prob = detect_language_langdetect(word_text)
        if lang != 'unknown':
            return lang, prob, 'langdetect'

    return 'english', 0.5, 'default'   # safe default


def _is_devanagari_visual(crop_gray):

    h, w = crop_gray.shape
    if h < 10:
        return False
    top_band  = crop_gray[:max(1, h // 5), :]   # top 20%
    rest_band = crop_gray[max(1, h // 5):, :]   # bottom 80%
    # Top band is darker (lower mean) and the difference is significant
    return (np.mean(rest_band) - np.mean(top_band) > 20
            and np.mean(top_band) < 220)


def detect_languages_for_all_words(word_results):

    for r in word_results:
        lang, conf, method = detect_word_language(
            r.get('text', ''),
            r.get('crop_gray', None)
        )
        r['lang']        = lang
        r['lang_conf']   = conf
        r['lang_method'] = method
    return word_results


# ── Demo: test language detection on sample strings ──────────
def _demo_language_detection():
    samples = [
        ('hello world',            'english'),
        ('मराठी ही महाराष्ट्राची भाषा', 'marathi'),
        ('தமிழ் மொழி இனிமையானது',  'tamil'),
    ]
    print("\\n[LANG DETECT] Demo:")
    for text, expected in samples:
        lang, conf, method = detect_word_language(text)
        status = '✅' if lang == expected else '❌'
        print(f"  {status} '{text[:30]:30s}'  → {lang:10s}  conf={conf:.2f}  [{method}]")

_demo_language_detection()
print("\\n[OK] Language detection module loaded.")

In [ ]:

# ============================================================
# SECTION 8 — OCR MODULE
# ============================================================
# ══════════════════════════════════════════════════════════════
#  OCR MODULE
#  Each language has its own function. Swap out any function
#  to plug in a better engine without touching the pipeline.
# ══════════════════════════════════════════════════════════════

# ── Tesseract config strings ──────────────────────────────────
_TESS_CONFIG_WORD = f'--psm {TESSERACT_PSM} --oem {TESSERACT_OEM}'
_TESS_LATIN_WHITELIST = (
    _TESS_CONFIG_WORD +
    ' -c tessedit_char_whitelist=ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz'
)


def _scale_crop_for_tesseract(crop_gray, min_side=50, scale_factor=2):

    h, w = crop_gray.shape
    if h < min_side or w < min_side:
        sh = max(2, min_side // max(1, h))
        sw = max(2, min_side // max(1, w))
        s  = max(sh, sw)
        return cv2.resize(crop_gray, (w * s, h * s), interpolation=cv2.INTER_CUBIC)
    return crop_gray


def english_ocr(crop_gray, binary_crop, cnn_model=None, charset=None):

    crop_scaled = _scale_crop_for_tesseract(crop_gray)
    pil_img     = PILImage.fromarray(crop_scaled)

    # ── Tesseract attempt ──────────────────────────────────────────────────
    try:
        text = pytesseract.image_to_string(
            pil_img, lang='eng', config=_TESS_LATIN_WHITELIST).strip()
        if text:
            return text, 0.75, 'tesseract-eng'
    except Exception as e:
        print(f"    [OCR WARN] Tesseract (eng) failed: {e}")

    # ── CNN fallback (single-character crops only) ─────────────────────────
    if cnn_model is not None and charset is not None:
        h, w = binary_crop.shape
        # WHY this guard: the CNN was trained on single characters (28×28 EMNIST).
        # Applying it to multi-character word crops gives meaningless results.
        if w <= h * CNN_FALLBACK_ASPECT_RATIO:
            resized = cv2.resize(binary_crop, (28, 28), interpolation=cv2.INTER_AREA)
            inp     = resized.reshape(1, 28, 28, 1).astype(np.float32) / 255.0
            prob    = cnn_model.predict(inp, verbose=0)[0]
            idx     = int(np.argmax(prob))
            return charset[idx], float(prob[idx]), 'cnn-fallback'
        else:
            return '[WORD]', 0.0, 'cnn-skipped-multichar'

    return '', 0.0, 'no-result'


def marathi_ocr(crop_gray, shirorekha_fix=True):

    crop_scaled = _scale_crop_for_tesseract(crop_gray, min_side=60)

    if shirorekha_fix:
        # Remove the horizontal top line before recognition
        crop_binary = cv2.adaptiveThreshold(
            crop_scaled, 255,
            cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 21, 5)
        crop_fixed = remove_shirorekha(crop_binary)
        pil_img = PILImage.fromarray(crop_fixed)
    else:
        pil_img = PILImage.fromarray(crop_scaled)

    # Try 'mar' pack first, then 'hin' as fallback (same Devanagari script)
    for lang_code in ['mar', 'hin']:
        try:
            text = pytesseract.image_to_string(
                pil_img, lang=lang_code, config=_TESS_CONFIG_WORD).strip()
            if text:
                return text, 0.65, f'tesseract-{lang_code}'
        except Exception as e:
            print(f"    [OCR WARN] Tesseract ({lang_code}) failed: {e}")

    return '?', 0.30, 'marathi-failed'


def tamil_ocr(crop_gray):

    crop_scaled = _scale_crop_for_tesseract(crop_gray, min_side=60)
    pil_img     = PILImage.fromarray(crop_scaled)

    try:
        text = pytesseract.image_to_string(
            pil_img, lang='tam', config=_TESS_CONFIG_WORD).strip()
        if text:
            return text, 0.65, 'tesseract-tam'
    except Exception as e:
        print(f"    [OCR WARN] Tesseract (tam) failed: {e}")

    return '?', 0.30, 'tamil-failed'


def run_ocr_for_word(crop_gray, binary_crop, language, cnn_model=None, charset=None):

    if language == 'english':
        return english_ocr(crop_gray, binary_crop, cnn_model, charset)
    elif language == 'marathi':
        return marathi_ocr(crop_gray, shirorekha_fix=True)
    elif language == 'tamil':
        return tamil_ocr(crop_gray)
    else:
        print(f"    [OCR WARN] Unknown language '{language}' — defaulting to English OCR")
        return english_ocr(crop_gray, binary_crop, cnn_model, charset)


print("[OK] OCR module loaded.")
print("  Functions: english_ocr | marathi_ocr | tamil_ocr | run_ocr_for_word")


In [ ]:
# ============================================================
# SECTION 9 — DATA & MODEL (corpus, augmentation, CNN, training)
# ============================================================
# ══════════════════════════════════════════════════════════════
#  MULTILINGUAL CORPUS LOADER
# ══════════════════════════════════════════════════════════════

LANGUAGES = {
    'en': ('wikipedia', '20220301.en'),
    'hi': ('wikipedia', '20220301.hi'),
    'mr': ('wikipedia', '20220301.mr'),
    'ta': ('wikipedia', '20220301.ta'),
    'te': ('wikipedia', '20220301.te'),
}

# Fallback sentences for when HuggingFace download fails
FALLBACK_CORPUS = {
    'en': ["the quick brown fox jumps over the lazy dog",
           "machine learning is transforming handwriting recognition",
           "neural networks learn complex patterns from data",
           "deep learning requires large amounts of training data",
           "convolutional neural networks excel at image classification"],
    'hi': ["यह एक अच्छा दिन है", "मशीन लर्निंग तेजी से विकसित हो रही है",
           "हिंदी भारत की राष्ट्रभाषा है", "शिक्षा सबसे महत्वपूर्ण है",
           "भारत एक विविधताओं वाला देश है"],
    'mr': ["मराठी ही महाराष्ट्राची अधिकृत भाषा आहे",
           "शिक्षण हे सर्वात महत्त्वाचे आहे",
           "मुंबई ही महाराष्ट्राची राजधानी आहे",
           "पुणे एक शैक्षणिक शहर आहे", "मराठी साहित्य खूप समृद्ध आहे"],
    'ta': ["தமிழ் மொழி இனிமையானது", "கல்வி மிகவும் முக்கியமானது",
           "இந்தியா ஒரு பெரிய நாடு", "செயற்கை நுண்ணறிவு வளர்ந்து வருகிறது",
           "தமிழ்நாடு ஒரு அழகான மாநிலம்"],
    'te': ["తెలుగు ఒక అందమైన భాష", "విద్య చాలా ముఖ్యమైనది",
           "భారతదేశం ఒక గొప్ప దేశం", "మెషిన్ లెర్నింగ్ వేగంగా పెరుగుతోంది",
           "తెలంగాణ ఒక కొత్త రాష్ట్రం"],
}


def _clean_sentence(text, min_words=5, max_words=30):

    text = re.sub(r'\\s+', ' ', text.strip())
    text = re.sub(r'[\\x00-\\x1f\\x7f-\\x9f]', '', text)
    text = re.sub(r'\\[.*?\\]|\\{.*?\\}|<.*?>', '', text)
    text = re.sub(r'={2,}|-{3,}', '', text).strip()
    words = text.split()
    if not (min_words <= len(words) <= max_words):
        return None
    if sum(c.isalpha() for c in text) / max(1, len(text)) < 0.6:
        return None
    return text


def _stream_lang(lang_code, config, n):

    from datasets import load_dataset
    sentences = []
    try:
        ds = load_dataset('wikipedia', config, split='train',
                          streaming=True, trust_remote_code=True)
        for sample in ds:
            for raw in sample['text'].split('. '):
                cleaned = _clean_sentence(raw)
                if cleaned:
                    sentences.append(cleaned)
                    if len(sentences) >= n:
                        return sentences
    except Exception as e:
        print(f"  [WARN] {lang_code} ({config}) failed: {e}")
    return sentences


def load_multilingual_corpus(target_per_lang=TARGET_PER_LANG, cache_path=DATA_CACHE):

    if cache_path.exists():
        print(f"[CORPUS] Loading cached corpus from {cache_path}")
        with open(cache_path, 'rb') as f:
            corpus = pickle.load(f)
        for lang, sents in corpus.items():
            print(f"  {lang:>3}: {len(sents):>4} sentences")
        return corpus

    print("[CORPUS] Downloading multilingual corpus (streaming)...")
    corpus = {}
    for lang_code, (dataset, config) in LANGUAGES.items():
        print(f"  Loading {lang_code}...", end=' ', flush=True)
        sents = _stream_lang(lang_code, config, target_per_lang)
        if len(sents) < 50:
            print(f"[FALLBACK] Using synthetic sentences for {lang_code}")
            pool  = FALLBACK_CORPUS.get(lang_code, FALLBACK_CORPUS['en'])
            sents = (pool * (target_per_lang // len(pool) + 1))[:target_per_lang]
        corpus[lang_code] = sents[:target_per_lang]
        print(f"{len(corpus[lang_code])} sentences")

    with open(cache_path, 'wb') as f:
        pickle.dump(corpus, f)
    print(f"[CORPUS] Corpus cached → {cache_path}")
    return corpus


MULTILANG_CORPUS = load_multilingual_corpus()

# ── Visualize corpus distribution ──────────────────────────────────────────
lang_labels = list(MULTILANG_CORPUS.keys())
lang_counts = [len(MULTILANG_CORPUS[l]) for l in lang_labels]
colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6']

fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(lang_labels, lang_counts, color=colors)
ax.set_title('Multilingual Corpus Distribution', fontsize=12, fontweight='bold')
ax.set_xlabel('Language'); ax.set_ylabel('Sentences')
for i, (l, c) in enumerate(zip(lang_labels, lang_counts)):
    ax.text(i, c + 5, str(c), ha='center', fontsize=9)
plt.tight_layout(); plt.show()
print(f"[CORPUS] Total: {sum(lang_counts)} sentences")

import albumentations as A

AUTOTUNE = tf.data.AUTOTUNE

# ── EMNIST Dataset ────────────────────────────────────────────
print("Downloading EMNIST/letters...")
(ds_train_raw, ds_test_raw), ds_info = tfds.load(
    'emnist/letters', split=['train', 'test'],
    as_supervised=True, with_info=True)

print(f"  Train: {ds_info.splits['train'].num_examples:,}")
print(f"  Test : {ds_info.splits['test'].num_examples:,}")

# ── Augmentation Pipeline ─────────────────────────────────────
# WHY these augmentations: handwriting varies in stroke angle (Rotate),
# ink pressure (BrightnessContrast), paper texture (GaussNoise),
# and pen path deformation (ElasticTransform).
aug_pipeline = A.Compose([
    A.ElasticTransform(alpha=1, sigma=10, alpha_affine=8, p=0.5),
    A.Perspective(scale=(0.03, 0.08), p=0.3),
    A.OneOf([
        A.GaussianBlur(blur_limit=(1, 3), p=1.0),
        A.MotionBlur(blur_limit=3, p=1.0),
    ], p=0.4),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.GaussNoise(var_limit=(5, 30), p=0.3),
    A.Rotate(limit=12, p=0.5, border_mode=cv2.BORDER_REPLICATE),
])


def numpy_augment(img_np):

    img_3ch  = np.repeat((img_np * 255).astype(np.uint8), 3, axis=-1)
    aug_out  = aug_pipeline(image=img_3ch)['image']
    return aug_out[..., :1].astype(np.float32) / 255.0


def tf_augment(image, label):

    aug_img = tf.numpy_function(func=numpy_augment, inp=[image], Tout=tf.float32)
    aug_img.set_shape([IMG_H, IMG_W, 1])
    return aug_img, label


def preprocess_emnist(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    image = tf.reshape(image, (IMG_H, IMG_W, 1))
    label = tf.cast(label - 1, tf.int32)   # 1-26 → 0-25
    return image, label


train_ds = (ds_train_raw
            .map(preprocess_emnist, num_parallel_calls=AUTOTUNE)
            .map(tf_augment, num_parallel_calls=AUTOTUNE)
            .shuffle(10_000, seed=SEED)
            .batch(BATCH_SIZE).prefetch(AUTOTUNE))

test_ds = (ds_test_raw
           .map(preprocess_emnist, num_parallel_calls=AUTOTUNE)
           .batch(BATCH_SIZE).cache().prefetch(AUTOTUNE))

# Cache test arrays for evaluation
print("Caching test arrays...")
X_test = np.concatenate([imgs.numpy() for imgs, _ in test_ds])
y_test = np.concatenate([lbls.numpy() for _, lbls in test_ds])
print(f"  X_test: {X_test.shape}, y_test: {y_test.shape}")

# ── DEBUG: Show augmented samples ─────────────────────────────
if DEBUG:
    fig, axes = plt.subplots(2, 8, figsize=(16, 4))
    for imgs, lbls in train_ds.take(1):
        for j in range(8):
            axes[0, j].imshow(X_test[j].squeeze(), cmap='gray')
            axes[0, j].set_title(f'Orig: {CHARSET[y_test[j]].upper()}', fontsize=8)
            axes[0, j].axis('off')
            axes[1, j].imshow(imgs[j].numpy().squeeze(), cmap='gray')
            axes[1, j].set_title(f'Aug: {CHARSET[lbls[j].numpy()].upper()}', fontsize=8)
            axes[1, j].axis('off')
    plt.suptitle('Original (top) vs Augmented (bottom)', fontweight='bold')
    plt.tight_layout(); plt.show()

print("[OK] Dataset and augmentation pipeline ready.")



In [ ]:
# ══════════════════════════════════════════════════════════════
#  RESNET CNN MODEL
# ══════════════════════════════════════════════════════════════

class LabelSmoothingLoss(keras.losses.Loss):

    def __init__(self, epsilon=0.1, num_classes=26, **kw):
        super().__init__(**kw)
        self.epsilon = epsilon; self.num_classes = num_classes

    def call(self, y_true, y_pred):
        y_pred   = tf.cast(y_pred, tf.float32)
        y_true_oh = tf.one_hot(tf.cast(y_true, tf.int32), self.num_classes)
        y_smooth  = (1 - self.epsilon) * y_true_oh + self.epsilon / self.num_classes
        return tf.reduce_mean(-tf.reduce_sum(y_smooth * tf.math.log(y_pred + 1e-8), axis=-1))

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'epsilon': self.epsilon, 'num_classes': self.num_classes})
        return cfg


def resnet_block(x, filters, stride=1, downsample=False):

    shortcut = x
    x = layers.Conv2D(filters, 3, strides=stride, padding='same',
                      kernel_regularizer=keras.regularizers.l2(1e-4), use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Conv2D(filters, 3, padding='same',
                      kernel_regularizer=keras.regularizers.l2(1e-4), use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    if downsample:
        shortcut = layers.Conv2D(filters, 1, strides=stride, padding='same',
                                 kernel_regularizer=keras.regularizers.l2(1e-4), use_bias=False)(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)
    x = layers.Add()([x, shortcut])
    return layers.ReLU()(x)


def build_resnet_cnn(num_classes=NUM_CLASSES):

    inp = keras.Input(shape=(IMG_H, IMG_W, 1), name='image_input')
    x = layers.Conv2D(32, 3, padding='same', kernel_regularizer=keras.regularizers.l2(1e-4), use_bias=False)(inp)
    x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
    x = resnet_block(x, 32);  x = resnet_block(x, 32)
    x = layers.MaxPooling2D(2)(x); x = layers.Dropout(0.20)(x)
    x = resnet_block(x, 64,  downsample=True); x = resnet_block(x, 64)
    x = layers.MaxPooling2D(2)(x); x = layers.Dropout(0.25)(x)
    x = resnet_block(x, 128, downsample=True); x = resnet_block(x, 128)
    x = layers.GlobalAveragePooling2D()(x); x = layers.Dropout(0.30)(x)
    x = layers.Dense(256, kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
    x = layers.Dropout(0.40)(x)
    out = layers.Dense(num_classes, activation='softmax', dtype='float32', name='predictions')(x)
    return Model(inp, out, name='ResNet_CNN')


model_cnn = build_resnet_cnn()
model_cnn.summary()
print(f"\\n[OK] Model parameters: {model_cnn.count_params():,}")
class WarmupCosineDecay(keras.optimizers.schedules.LearningRateSchedule):

    def __init__(self, peak_lr=1e-3, min_lr=1e-5, warmup_steps=500, total_steps=10000, **kw):
        super().__init__(**kw)
        self.peak_lr = float(peak_lr); self.min_lr = float(min_lr)
        self.warmup_steps = float(warmup_steps); self.total_steps = float(total_steps)

    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        warmup_lr = self.peak_lr * (step / self.warmup_steps)
        cos_lr    = self.min_lr + 0.5 * (self.peak_lr - self.min_lr) * (
                    1.0 + tf.cos(np.pi * (step - self.warmup_steps) /
                                 (self.total_steps - self.warmup_steps)))
        return tf.where(step < self.warmup_steps, warmup_lr, cos_lr)

    def get_config(self):
        return {'peak_lr': self.peak_lr, 'min_lr': self.min_lr,
                'warmup_steps': self.warmup_steps, 'total_steps': self.total_steps}


class EpochTimer(keras.callbacks.Callback):

    def __init__(self):
        super().__init__(); self.epoch_times = []; self.total_start = None
    def on_train_begin(self, logs=None):   self.total_start = time.time()
    def on_epoch_begin(self, epoch, logs=None): self._t0 = time.time()
    def on_epoch_end(self, epoch, logs=None):
        elapsed = time.time() - self._t0; self.epoch_times.append(elapsed)
        print(f"  ⏱  Epoch {epoch+1} time: {elapsed:.1f}s")
    def on_train_end(self, logs=None):
        self.total_time = time.time() - self.total_start
        print(f"\\n[TIME] Total training: {self.total_time:.1f}s ({self.total_time/60:.1f} min)")


def load_or_train_model():

    custom_objects = {'LabelSmoothingLoss': LabelSmoothingLoss, 'WarmupCosineDecay': WarmupCosineDecay}
    ckpt_keras = Path(str(CNN_CKPT) + '.keras')
    if ckpt_keras.exists():
        print(f"[CKPT] Loading model from {ckpt_keras}...")
        try:
            m = keras.models.load_model(str(ckpt_keras), custom_objects=custom_objects)
            print("[CKPT] Model loaded — skipping training.")
            history_dict = {}
            if HIST_FILE.exists():
                with open(HIST_FILE, 'rb') as f: history_dict = pickle.load(f)
            return m, history_dict, None
        except Exception as e:
            print(f"[CKPT] Load failed ({e}) — retraining.")

    total_steps = CNN_EPOCHS * (ds_info.splits['train'].num_examples // BATCH_SIZE)
    lr_schedule = WarmupCosineDecay(peak_lr=1e-3, min_lr=1e-5,
                                    warmup_steps=WARMUP_STEPS, total_steps=total_steps)
    model_cnn.compile(
        optimizer=keras.optimizers.Adam(lr_schedule, clipnorm=1.0),
        loss=LabelSmoothingLoss(epsilon=0.1, num_classes=NUM_CLASSES),
        metrics=['accuracy'])

    timer_cb = EpochTimer()
    callbacks = [
        timer_cb,
        keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True, verbose=1),
        keras.callbacks.ModelCheckpoint(filepath=str(ckpt_keras), save_best_only=True,
                                        monitor='val_accuracy', verbose=1),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3,
                                          min_lr=1e-6, verbose=1),
    ]
    print(f"[TRAIN] Training ResNet CNN for up to {CNN_EPOCHS} epochs...")
    history = model_cnn.fit(train_ds, epochs=CNN_EPOCHS,
                            validation_data=test_ds, callbacks=callbacks, verbose=1)
    history_dict = history.history
    with open(HIST_FILE, 'wb') as f: pickle.dump(history_dict, f)
    return model_cnn, history_dict, timer_cb


model_cnn, cnn_history_dict, timer_cb = load_or_train_model()
cnn_loss, cnn_acc = model_cnn.evaluate(test_ds, verbose=0)
print(f"\\n[EVAL] CNN Test Accuracy : {cnn_acc*100:.2f}%")
print(f"[EVAL] CNN Test Loss     : {cnn_loss:.4f}")


In [ ]:
# ============================================================
# SECTION 10 — PIPELINE INTEGRATION
# ============================================================
# ══════════════════════════════════════════════════════════════
#  PIPELINE INTEGRATION
# ══════════════════════════════════════════════════════════════

STOP_WORDS  = set(stopwords.words('english'))
lemmatizer  = WordNetLemmatizer()
LANG_NAMES  = {
    'en': 'English', 'hi': 'Hindi', 'mr': 'Marathi',
    'ta': 'Tamil',   'te': 'Telugu', 'unknown': 'Unknown'
}


def _translate_to_english(text, src_lang):

    if src_lang == 'en' or src_lang == 'english':
        return text
    try:
        return GoogleTranslator(source='auto', target='en').translate(text)
    except Exception as e:
        print(f"  [TRANSLATE WARN] {e}")
        return text


def run_nlp_pipeline_on_text(text, verbose=True):

    from transformers import pipeline as hf_pipeline
    # Lazy-load the sentiment model (avoid reloading on every call)
    if not hasattr(run_nlp_pipeline_on_text, '_sentiment_pipe'):
        print("  [NLP] Loading multilingual sentiment model...")
        try:
            run_nlp_pipeline_on_text._sentiment_pipe = hf_pipeline(
                'sentiment-analysis',
                model='nlptown/bert-base-multilingual-uncased-sentiment',
                truncation=True, max_length=512,
                device=0 if tf.config.list_physical_devices('GPU') else -1)
        except Exception:
            run_nlp_pipeline_on_text._sentiment_pipe = hf_pipeline(
                'sentiment-analysis',
                model='distilbert-base-uncased-finetuned-sst-2-english',
                truncation=True, max_length=512)

    sentiment_pipe = run_nlp_pipeline_on_text._sentiment_pipe

    def _normalise_sentiment(raw):
        label = raw['label'].upper(); score = raw['score']
        if 'STAR' in label:
            stars = int(label.split()[0])
            polarity = 'POSITIVE' if stars >= 4 else ('NEGATIVE' if stars <= 2 else 'NEUTRAL')
        elif label in ('POSITIVE', 'LABEL_1'): polarity = 'POSITIVE'
        elif label in ('NEGATIVE', 'LABEL_0'): polarity = 'NEGATIVE'
        else:                                  polarity = 'NEUTRAL'
        return {'label': polarity, 'confidence': round(score * 100, 2)}

    doc      = nlp_spacy(text)
    entities = [(ent.text, ent.label_) for ent in doc.ents]
    pos_tags = [(t.text, t.pos_) for t in doc]
    raw_sent = sentiment_pipe(text[:512])[0]
    sentiment = _normalise_sentiment(raw_sent)

    if verbose:
        emoji = ':)' if sentiment['label'] == 'POSITIVE' else (':|' if sentiment['label'] == 'NEUTRAL' else ':/')
        print(f"  [NLP] Entities  : {entities if entities else 'None'}")
        print(f"  [NLP] Sentiment : {sentiment['label']} {emoji} ({sentiment['confidence']}%)")
    return {'entities': entities, 'pos_tags': pos_tags, 'sentiment': sentiment}


def run_full_pipeline(image_path, debug=None, apply_shirorekha_fix=False):

    if debug is None:
        debug = DEBUG

    SEP = '═' * 65
    print(f"\\n{SEP}")
    print(f"  🧠 FULL HTR + NLP PIPELINE")
    print(f"  Image: {os.path.basename(str(image_path))}")
    print(SEP)

    # ── STAGE 1: Preprocessing ───────────────────────────────────────────────
    print("\\n▶ STAGE 1 — Preprocessing")
    preprocessed = preprocess_image(image_path, debug=debug)
    binary        = preprocessed['binary']
    img_orig      = preprocessed['upscaled']

    if debug:
        show_image('Input Image (upscaled grayscale)', img_orig)

    # ── STAGE 2: Line Segmentation ───────────────────────────────────────────
    print("\\n▶ STAGE 2 — Line Segmentation")
    binary_for_seg = binary
    if apply_shirorekha_fix:
        print("  Applying Devanagari shirorekha fix to binary image...")
        binary_for_seg = remove_shirorekha(binary)
        if debug:
            show_before_after('Binary (original)', binary, 'Binary (shirorekha removed)', binary_for_seg)

    line_boxes, line_intermediates = find_text_lines(binary_for_seg, debug=debug)
    print(f"  → Found {len(line_boxes)} text lines")

    # ── STAGE 3 & 4: Word Segmentation + Language Detection + OCR ────────────
    print("\\n▶ STAGE 3 — Word Segmentation (dual method)")
    print("▶ STAGE 4 — Language Detection + OCR (per word)")

    all_word_results = []   # Will hold one dict per detected word
    line_texts       = []   # Reconstructed text per line

    for line_idx, (lx1, ly1, lx2, ly2) in enumerate(line_boxes):
        print(f"\\n  Line {line_idx + 1}/{len(line_boxes)}:")

        # Extract line crops from the binary and gray images
        line_bin  = binary_for_seg[ly1:ly2, lx1:lx2]
        line_gray = img_orig[ly1:ly2, lx1:lx2]

        if debug:
            show_image(f'Line {line_idx + 1} crop (binary)', line_bin)
            show_projection_profile(line_bin, axis='x',
                                    title=f'Line {line_idx + 1} — X-Projection (word gaps)')

        # ── Method A: Connected components ───────────────────────────────────
        cc_boxes = word_seg_connected_components(line_bin, debug=debug)
        print(f"    CC method found {len(cc_boxes)} word box(es)")

        # ── Method B: X-projection ───────────────────────────────────────────
        xp_boxes = word_seg_xprojection(line_bin, debug=debug)
        print(f"    X-proj method found {len(xp_boxes)} word box(es)")

        # ── Merge the two sets ────────────────────────────────────────────────
        word_boxes_local = merge_word_boxes(cc_boxes, xp_boxes)
        print(f"    After merge: {len(word_boxes_local)} word box(es)")

        # Convert line-relative coordinates to image-absolute + add padding
        line_word_results = []
        for (wx1, wy1, wx2, wy2) in sorted(word_boxes_local, key=lambda b: b[0]):
            # Absolute coordinates in the full image
            ax1 = max(0,              lx1 + wx1 - WORD_PADDING_PX)
            ay1 = max(0,              ly1 + wy1 - WORD_PADDING_PX)
            ax2 = min(img_orig.shape[1], lx1 + wx2 + WORD_PADDING_PX)
            ay2 = min(img_orig.shape[0], ly1 + wy2 + WORD_PADDING_PX)

            crop_gray   = img_orig[ay1:ay2, ax1:ax2]
            crop_binary = binary[ay1:ay2, ax1:ax2]   # use original binary for OCR

            # ── Language detection (pre-OCR, visual heuristic only) ───────────
            lang_pre, _, lang_method = detect_word_language('', crop_gray)

            # ── Language-specific OCR ─────────────────────────────────────────
            text, conf, ocr_method = run_ocr_for_word(
                crop_gray, crop_binary, lang_pre,
                cnn_model=model_cnn, charset=CHARSET)

            # ── Post-OCR language detection (uses OCR text too) ───────────────
            lang_final, lang_conf, lang_method2 = detect_word_language(text, crop_gray)

            word_entry = {
                'text':        text,
                'conf':        conf,
                'ocr_method':  ocr_method,
                'lang':        lang_final,
                'lang_conf':   lang_conf,
                'lang_method': lang_method2,
                'bbox':        (ax1, ay1, ax2, ay2),
                'crop_gray':   crop_gray,
                'crop_binary': crop_binary,
                'line_idx':    line_idx,
            }
            line_word_results.append(word_entry)
            all_word_results.append(word_entry)

        # ── Debug: Show word bounding boxes for this line ─────────────────────
        if debug and line_word_results:
            show_language_boxes(
                img_orig,
                line_word_results,
                figsize=(12, 4)
            )
            show_word_crops(
                [r['crop_gray'] for r in line_word_results],
                predictions=line_word_results,
                title=f'Line {line_idx + 1} Word Crops + Predictions'
            )

        # Reconstruct line text
        line_text = ' '.join(
            r['text'] for r in line_word_results if r['text'].strip()
        )
        line_texts.append(line_text)
        print(f"    Line OCR: \"{line_text}\"")

    # ── STAGE 5: Post-processing ─────────────────────────────────────────────
    print("\\n▶ STAGE 5 — Post-Processing (Reconstruction + NLP)")

    raw_text = '\\n'.join(line_texts)
    print(f"\\n  Reconstructed text:\\n  {repr(raw_text)}")

    # Detect dominant language across all words
    lang_votes = Counter(r['lang'] for r in all_word_results)
    dominant_lang = lang_votes.most_common(1)[0][0] if lang_votes else 'english'
    print(f"  Dominant language: {dominant_lang}  (votes: {dict(lang_votes)})")

    # Translate to English for NLP if needed
    lang_code_map = {'english': 'en', 'marathi': 'mr', 'tamil': 'ta'}
    src_code      = lang_code_map.get(dominant_lang, 'en')
    english_text  = _translate_to_english(raw_text, src_code)
    if english_text != raw_text:
        print(f"  Translated: {repr(english_text)}")

    # Run NLP
    nlp_out = {}
    if english_text.strip():
        nlp_out = run_nlp_pipeline_on_text(english_text, verbose=True)

    # ── Build final pipeline state dict ──────────────────────────────────────
    pipeline_state = {
        'preprocessed':  preprocessed,
        'line_boxes':    line_boxes,
        'word_results':  all_word_results,
        'raw_text':      raw_text,
        'language':      dominant_lang,
        'english_text':  english_text,
        'nlp':           nlp_out,
    }

    print(f"\\n{SEP}")
    print(f"  ✅ PIPELINE COMPLETE")
    print(f"  Words detected : {len(all_word_results)}")
    print(f"  Lines detected : {len(line_boxes)}")
    print(f"  Language       : {dominant_lang}")
    sent = nlp_out.get('sentiment', {})
    print(f"  Sentiment      : {sent.get('label','N/A')} ({sent.get('confidence',0)}%)")
    print(SEP)

    return pipeline_state


print("[OK] Pipeline integration module loaded.")
print("  Entry point: run_full_pipeline(image_path, debug=True)")



In [ ]:
# ============================================================
# SECTION 11 — RUN THE PIPELINE (upload cell)
# ============================================================
# ══════════════════════════════════════════════════════════════
#  UPLOAD YOUR IMAGE AND RUN THE FULL PIPELINE
# ══════════════════════════════════════════════════════════════

try:
    from google.colab import files
    print('[FILE] Upload a handwritten image (jpg / png)...')
    uploaded = files.upload()
    if uploaded:
        img_path = list(uploaded.keys())[0]
        print(f'[OK] Uploaded: {img_path}')
    else:
        img_path = None
except ImportError:
    # Running outside Colab — specify path manually
    img_path = 'images.png'

if img_path and os.path.exists(str(img_path)):
    # ── Run the complete pipeline ─────────────────────────────────────────
    pipeline_state = run_full_pipeline(
        image_path=img_path,
        debug=True,                    # Set False to suppress intermediate visuals
        apply_shirorekha_fix=False,    # Set True if image contains Marathi/Hindi
    )
else:
    print('[WARN] No image found. Running with demo text (no image segmentation).')
    pipeline_state = {
        'word_results': [], 'raw_text': 'The quick brown fox jumps over the lazy dog',
        'language': 'english', 'english_text': 'The quick brown fox jumps over the lazy dog',
        'line_boxes': [], 'preprocessed': {}, 'nlp': {}
    }


In [ ]:
# ============================================================
# SECTION 12 — DEBUG PANEL
# ============================================================
# ══════════════════════════════════════════════════════════════
#  DEBUG PANEL — Complete Inspection of Pipeline Results
# ══════════════════════════════════════════════════════════════

def show_debug_panel(pipeline_state):

    pre     = pipeline_state.get('preprocessed', {})
    words   = pipeline_state.get('word_results', [])
    lines   = pipeline_state.get('line_boxes', [])

    print("\\n" + "═"*65)
    print("  🔍 DEBUG PANEL")
    print("═"*65)

    # ── Panel 1: Preprocessing stages ─────────────────────────────────────
    if 'upscaled' in pre and 'binary' in pre:
        print("\\n📌 Panel 1: Preprocessing Stages")
        show_multi_stage([
            ('Original (gray)',       pre.get('upscaled', np.zeros((10,10)))),
            ('CLAHE Enhanced',        pre.get('enhanced', np.zeros((10,10)))),
            ('Bilateral Denoised',    pre.get('denoised', np.zeros((10,10)))),
            ('Binary (inverted)',     pre.get('binary',   np.zeros((10,10)))),
        ], figsize=(20, 3))

    # ── Panel 2: Line bounding boxes on the original ──────────────────────
    if lines and 'upscaled' in pre:
        print(f"\\n📌 Panel 2: Line Segmentation ({len(lines)} lines)")
        show_bounding_boxes(
            pre['upscaled'], lines,
            color=(0, 200, 255), thickness=3,
            labels=[f'L{i+1}' for i in range(len(lines))],
            title=f'Text Lines Detected ({len(lines)})'
        )

    # ── Panel 3: All word bounding boxes ─────────────────────────────────
    if words and 'upscaled' in pre:
        print(f"\\n📌 Panel 3: Word Bounding Boxes ({len(words)} words)")
        all_boxes  = [r['bbox'] for r in words]
        all_labels = [f"{r['text'][:6]}" for r in words]
        show_bounding_boxes(
            pre['upscaled'], all_boxes,
            color=(0, 200, 0), thickness=2,
            labels=all_labels,
            title=f'All Word Boxes ({len(words)})'
        )

    # ── Panel 4: Language-colour-coded boxes ─────────────────────────────
    if words and 'upscaled' in pre:
        print("\\n📌 Panel 4: Language Detection (colour-coded)")
        show_language_boxes(pre['upscaled'], words)

    # ── Panel 5: Word crops strip (all words) ────────────────────────────
    if words:
        print(f"\\n📌 Panel 5: All Word Crops (showing up to 20)")
        show_word_crops(
            [r['crop_gray'] for r in words],
            predictions=words,
            max_show=20,
            title='All Detected Word Crops'
        )

    # ── Panel 6: Per-word text table ─────────────────────────────────────
    print("\\n📌 Panel 6: Per-Word Prediction Table")
    print(f"  {'#':>3}  {'Text':20s}  {'Conf':5s}  {'Lang':10s}  {'OCR Method':22s}  {'BBox'}")
    print("  " + "-"*85)
    for i, r in enumerate(words):
        bar  = '█' * int(r['conf'] * 10) + '░' * (10 - int(r['conf'] * 10))
        bbox = r['bbox']
        print(f"  {i:>3}  {r['text'][:20]:20s}  {r['conf']:.2f}  "
              f"{r['lang']:10s}  {r['ocr_method']:22s}  {bbox}")

    # ── Panel 7: Failure analysis ─────────────────────────────────────────
    failures = [r for r in words if r['conf'] < 0.40 or r['text'] in ('', '?', '[WORD]')]
    print(f"\\n📌 Panel 7: Low-Confidence / Failed Words ({len(failures)} of {len(words)})")
    if failures:
        show_word_crops(
            [r['crop_gray'] for r in failures],
            predictions=failures,
            max_show=12,
            title=f'⚠️ Failed / Low-Confidence Words'
        )
        for r in failures:
            print(f"  ⚠️  text='{r['text']}'  conf={r['conf']:.2f}  method={r['ocr_method']}  bbox={r['bbox']}")
    else:
        print("  ✅ No failures detected!")

    # ── Panel 8: Final output ─────────────────────────────────────────────
    print("\\n📌 Panel 8: Final Reconstructed Output")
    print(f"  Raw OCR text  : {pipeline_state.get('raw_text', '')!r}")
    print(f"  Language      : {pipeline_state.get('language', 'N/A')}")
    print(f"  English text  : {pipeline_state.get('english_text', '')!r}")
    nlp = pipeline_state.get('nlp', {})
    sent = nlp.get('sentiment', {})
    print(f"  Sentiment     : {sent.get('label','N/A')} ({sent.get('confidence', 0)}%)")
    print(f"  Entities      : {nlp.get('entities', [])}")
    print("\\n" + "═"*65)


# Run the debug panel
if 'pipeline_state' in locals():
    show_debug_panel(pipeline_state)
else:
    print("[WARN] No pipeline_state found — run Section 11 first.")


In [ ]:
## 📊 Section 13 — Evaluation: CER / WER / Confusion Matrix
# ══════════════════════════════════════════════════════════════
#  EVALUATION MODULE
# ══════════════════════════════════════════════════════════════

def compute_cer_wer(predictions, ground_truths):

    assert len(predictions) == len(ground_truths), "Length mismatch"
    cer_list, wer_list, details = [], [], []
    for pred, truth in zip(predictions, ground_truths):
        c_dist = editdistance.eval(list(pred), list(truth))
        cer    = c_dist / max(1, len(truth))
        w_dist = editdistance.eval(pred.split(), truth.split())
        wer    = w_dist / max(1, len(truth.split()))
        cer_list.append(cer); wer_list.append(wer)
        details.append({'pred': pred, 'truth': truth, 'cer': cer, 'wer': wer})

    results = {'mean_cer': np.mean(cer_list), 'mean_wer': np.mean(wer_list), 'details': details}
    print(f"  Mean CER : {results['mean_cer']*100:.2f}%")
    print(f"  Mean WER : {results['mean_wer']*100:.2f}%")
    return results


# ── CNN character-level evaluation ────────────────────────────
y_pred_prob = model_cnn.predict(X_test, batch_size=256, verbose=0)
y_pred_cnn  = np.argmax(y_pred_prob, axis=1)

gt_chars   = [CHARSET[i] for i in y_test[:2000]]
pred_chars = [CHARSET[i] for i in y_pred_cnn[:2000]]
print("\\n[EVAL] CNN Single-character CER/WER (2000 samples):")
ocr_results = compute_cer_wer(pred_chars, gt_chars)

# ── Per-class accuracy ─────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred_cnn)
per_class_acc = cm.diagonal() / cm.sum(axis=1)
sorted_idx    = np.argsort(per_class_acc)

fig, axes = plt.subplots(1, 2, figsize=(20, 7))

# Confusion matrix
sns.heatmap(cm, annot=False, fmt='d', cmap='Blues',
            xticklabels=[c.upper() for c in CHARSET],
            yticklabels=[c.upper() for c in CHARSET],
            ax=axes[0], linewidths=0)
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')
axes[0].set_title('Confusion Matrix — ResNet CNN', fontsize=14, fontweight='bold')

# Per-class bar chart
colors_ = ['#e74c3c' if per_class_acc[i] < 0.85 else '#2ecc71' for i in sorted_idx]
axes[1].bar([CHARSET[i].upper() for i in sorted_idx],
            [per_class_acc[i]*100 for i in sorted_idx], color=colors_)
axes[1].axhline(np.mean(per_class_acc)*100, color='navy', linestyle='--', linewidth=2,
                label=f'Mean: {np.mean(per_class_acc)*100:.1f}%')
axes[1].set_xlabel('Letter'); axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Per-Class Accuracy (red = below 85%)', fontsize=12, fontweight='bold')
axes[1].legend(); axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('CNN Evaluation', fontsize=15, fontweight='bold')
plt.tight_layout(); plt.show()

worst5 = [(CHARSET[i].upper(), f'{per_class_acc[i]*100:.1f}%') for i in sorted_idx[:5]]
best5  = [(CHARSET[i].upper(), f'{per_class_acc[i]*100:.1f}%') for i in sorted_idx[-5:][::-1]]
print(f"5 Hardest: {worst5}")
print(f"5 Easiest: {best5}")
"""))

cells.append(code("""# ── Error analysis: show top misclassified samples ────────────
wrong_mask = (y_pred_cnn != y_test)
wrong_idx  = np.where(wrong_mask)[0]
print(f"\\n[ERROR ANALYSIS] Total errors: {len(wrong_idx)} / {len(y_test)} "
      f"({len(wrong_idx)/len(y_test)*100:.1f}%)")

# Most confused pairs
confusion_pairs = Counter(
    (CHARSET[y_test[i]].upper(), CHARSET[y_pred_cnn[i]].upper())
    for i in wrong_idx)
print("\\nTop 10 confusion pairs (True → Predicted):")
for (true_ch, pred_ch), count in confusion_pairs.most_common(10):
    print(f"  {true_ch} → {pred_ch}: {count} times")

# ── Visual: 16 worst errors ─────────────────────────────────
n_show = min(16, len(wrong_idx))
fig, axes = plt.subplots(2, 8, figsize=(16, 5))
axes = axes.ravel()
for ax_i, idx in enumerate(wrong_idx[:n_show]):
    conf = float(y_pred_prob[idx].max())
    axes[ax_i].imshow(X_test[idx].squeeze(), cmap='gray')
    axes[ax_i].set_title(
        f"GT:{CHARSET[y_test[idx]].upper()}\\nP:{CHARSET[y_pred_cnn[idx]].upper()} ({conf:.2f})",
        fontsize=7, color='red')
    axes[ax_i].axis('off')
    # Red border to make errors visually obvious
    for spine in axes[ax_i].spines.values():
        spine.set_edgecolor('red'); spine.set_linewidth(3)
for ax_i in range(n_show, len(axes)):
    axes[ax_i].axis('off')
plt.suptitle('⚠️ Top Misclassified Samples (GT → Predicted, confidence)',
             fontweight='bold', color='darkred')
plt.tight_layout(); plt.show()

# ── Confidence distribution ──────────────────────────────────
correct_conf = y_pred_prob[~wrong_mask].max(axis=1)
wrong_conf   = y_pred_prob[wrong_mask].max(axis=1)
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(correct_conf, bins=50, alpha=0.7, label='Correct', color='steelblue', density=True)
ax.hist(wrong_conf,   bins=50, alpha=0.7, label='Incorrect', color='coral',    density=True)
ax.set_xlabel('Confidence'); ax.set_ylabel('Density')
ax.set_title('Confidence Distribution: Correct vs Incorrect Predictions', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()
print(f"Mean confidence — Correct: {correct_conf.mean():.3f} | Incorrect: {wrong_conf.mean():.3f}")

# ══════════════════════════════════════════════════════════════
#  PERFORMANCE SUMMARY DASHBOARD
# ══════════════════════════════════════════════════════════════

print('═'*65)
print('  📊 SYSTEM PERFORMANCE SUMMARY')
print('═'*65)
print(f'  CNN Architecture   : ResNet-style (residual blocks + BatchNorm)')
print(f'  Parameters         : {model_cnn.count_params():,}')
print(f'  Test Accuracy      : {cnn_acc*100:.2f}%')
print(f'  Test Loss          : {cnn_loss:.4f}')
print(f'  OCR Mean CER       : {ocr_results["mean_cer"]*100:.2f}%')
print(f'  OCR Mean WER       : {ocr_results["mean_wer"]*100:.2f}%')
if timer_cb and hasattr(timer_cb, 'total_time'):
    print(f'  Training Time      : {timer_cb.total_time:.0f}s ({timer_cb.total_time/60:.1f} min)')
    print(f'  Avg per Epoch      : {np.mean(timer_cb.epoch_times):.1f}s')
else:
    print(f'  Training Time      : loaded from checkpoint')
print(f'  Languages Supported: English, Marathi (Devanagari), Tamil')
print(f'  Checkpoint Path    : {CNN_CKPT}')
print('═'*65)
print()
print('  Techniques applied:')
for item in [
    'ResNet residual blocks + skip connections',
    'Label smoothing (ε=0.1) for overconfidence prevention',
    'L2 weight decay on all conv/dense layers',
    'Mixed precision training (float16)',
    'WarmupCosineDecay LR schedule',
    'Gradient clipping (clipnorm=1.0)',
    'Elastic distortion + perspective augmentation',
    'Dual-method word segmentation (CC + X-projection)',
    'Unicode heuristic + langdetect language detection',
    'Script-routed OCR (English / Marathi / Tamil)',
    'Devanagari shirorekha removal preprocessing',
    'CER + WER evaluation',
    'Per-word debug panel with failure visualization',
]:
    print(f'  ✅ {item}')
print('═'*65)
